# Python Drills, Part 1: Functions, Lists, Loops, Dictionaries (SOLUTIONS)

This notebook is practice with the core Python you need for everything else in this course. None of it is
machine learning yet, but almost every bug you will hit in an ML notebook comes from one of these basics.

How it works:

- Each section starts with a short explanation and some example cells. Run the examples and change them
  to see what happens; that is the fastest way to learn what a line does.
- Exercise cells contain a function stub marked `# YOUR CODE HERE`. Replace the `pass` with your code.
- Right below each exercise is a test cell. Run it after your exercise cell. Each test prints `PASS` or
  `FAIL`, and a failure shows what was expected and what your code produced.
- Do not edit the test cells. If a test seems wrong, ask.
- Unless an exercise says otherwise, do not use NumPy or any imports in this notebook. Part 2 is where
  NumPy comes in.
- Before you submit, do **Runtime > Restart session and run all** and check the summary at the very end.

Run the setup cell below first. It defines the `check` function that all of the tests use.

In [1]:
# Setup: run this cell first. You do not need to read or change it.
import math, numbers

_results = {}

def _same(got, expected, tol):
    if isinstance(expected, (list, tuple)) and isinstance(got, (list, tuple)):
        return len(got) == len(expected) and all(_same(g, e, tol) for g, e in zip(got, expected))
    if isinstance(expected, dict) and isinstance(got, dict):
        return got.keys() == expected.keys() and all(_same(got[k], expected[k], tol) for k in expected)
    if (isinstance(expected, numbers.Number) and isinstance(got, numbers.Number)
            and not isinstance(expected, bool) and not isinstance(got, bool)):
        return math.isclose(got, expected, rel_tol=tol, abs_tol=tol)
    return got == expected

def check(name, got, expected, tol=1e-6):
    """Compare got to expected and print PASS or FAIL. If got is a function, it is called first."""
    if callable(got):
        try:
            got = got()
        except Exception as e:
            _results[name] = False
            print(f"FAIL  {name}\n        raised {type(e).__name__}: {e}")
            return
    ok = _same(got, expected, tol)
    _results[name] = ok
    if ok:
        print(f"PASS  {name}")
    else:
        print(f"FAIL  {name}\n        expected: {expected!r}\n        got:      {got!r}")
        if got is None:
            print("        (got None: is the function still a stub, or did you forget to return?)")

def check_raises(name, func, exc_type):
    """Pass if calling func() raises exc_type."""
    try:
        func()
    except exc_type:
        ok, msg = True, ""
    except Exception as e:
        ok, msg = False, f"raised {type(e).__name__} instead of {exc_type.__name__}"
    else:
        ok, msg = False, f"did not raise {exc_type.__name__}"
    _results[name] = ok
    print(f"PASS  {name}" if ok else f"FAIL  {name}\n        {msg}")

def report():
    passed = sum(_results.values())
    print(f"{passed} / {len(_results)} tests passed")
    for name, ok in _results.items():
        if not ok:
            print("  failing:", name)

print("Setup done.")

Setup done.


A note on reading the tests: some tests look like `check("...", lambda: some_function(3), 7)`. The
`lambda:` just wraps the call so that if your code crashes, the test reports a FAIL instead of stopping
the notebook. You do not need to know anything else about `lambda` for now.

## Section 1: Functions

A function is a named piece of code that takes inputs (parameters) and hands back an output with
`return`. The first line of the body can be a string describing what the function does, called a
docstring.

In [2]:
def rectangle_area(width, height):
    """Return the area of a width-by-height rectangle."""
    return width * height

print(rectangle_area(3, 4))
a = rectangle_area(2.5, 2)
print(a)

12
5.0


**`return` is not the same as `print`.** This is probably the single most common bug in student code.
`print` shows a value on the screen and then throws it away. `return` hands the value back to whoever
called the function so it can be stored or used. A function with no `return` statement returns `None`.

In [3]:
def add_print(a, b):
    print(a + b)

def add_return(a, b):
    return a + b

x = add_print(2, 3)    # this prints 5...
print("x is", x)       # ...but x is None
y = add_return(2, 3)   # prints nothing
print("y is", y)       # y is 5, and we can keep using it
print(add_return(2, 3) * 10)

5
x is None
y is 5
50


**Default parameters** let a caller leave an argument out. Parameters with defaults must come after
the ones without. You can also pass any argument by name (a *keyword argument*), in which case the
order does not matter.

In [4]:
def greet(name, greeting="Hello", punctuation="!"):
    return f"{greeting}, {name}{punctuation}"

print(greet("Ada"))
print(greet("Ada", "Hi"))
print(greet("Ada", punctuation="?"))
print(greet(greeting="Yo", name="Ada"))

Hello, Ada!
Hi, Ada!
Hello, Ada?
Yo, Ada!


**Returning more than one value.** Separate the values with commas and Python packs them into a
tuple. The caller can unpack the tuple into separate variables.

In [5]:
def div_and_mod(a, b):
    return a // b, a % b

result = div_and_mod(17, 5)
print(result, type(result))
q, r = div_and_mod(17, 5)
print("quotient", q, "remainder", r)

(3, 2) <class 'tuple'>
quotient 3 remainder 2


**Conditionals** inside functions: `if`, `elif`, `else`. Once a `return` runs, the function is over, so
you often do not need an `else` after a branch that returns.

In [6]:
def describe(n):
    if n < 0:
        return "negative"
    elif n == 0:
        return "zero"
    return "positive"

print(describe(-4), describe(0), describe(10))

negative zero positive


### Exercise 1.1: `c_to_f`
Write a function that converts Celsius to Fahrenheit: $F = \frac{9}{5}C + 32$.

In [7]:
def c_to_f(c):
    """Convert a temperature in Celsius to Fahrenheit."""
    return c * 9 / 5 + 32

In [8]:
check("1.1 c_to_f(0)", c_to_f(0), 32)
check("1.1 c_to_f(100)", c_to_f(100), 212)
check("1.1 c_to_f(-40)", c_to_f(-40), -40)
check("1.1 c_to_f(37)", c_to_f(37), 98.6)

PASS  1.1 c_to_f(0)
PASS  1.1 c_to_f(100)
PASS  1.1 c_to_f(-40)
PASS  1.1 c_to_f(37)


### Exercise 1.2: `clip`
Write `clip(x, lo=0, hi=1)`. If `x` is below `lo` return `lo`, if it is above `hi` return `hi`, and
otherwise return `x` unchanged. Do not use the built-in `min` or `max`; use `if`.

In [9]:
def clip(x, lo=0, hi=1):
    """Return x limited to the range [lo, hi]."""
    if x < lo:
        return lo
    if x > hi:
        return hi
    return x

In [10]:
check("1.2 clip(0.5)", clip(0.5), 0.5)
check("1.2 clip(-3)", clip(-3), 0)
check("1.2 clip(7)", clip(7), 1)
check("1.2 clip(7, hi=10)", clip(7, hi=10), 7)
check("1.2 clip(-3, lo=-5, hi=5)", clip(-3, lo=-5, hi=5), -3)
check("1.2 clip(12, 0, 10)", clip(12, 0, 10), 10)

PASS  1.2 clip(0.5)
PASS  1.2 clip(-3)
PASS  1.2 clip(7)
PASS  1.2 clip(7, hi=10)
PASS  1.2 clip(-3, lo=-5, hi=5)
PASS  1.2 clip(12, 0, 10)


### Exercise 1.3: `distance`
Write `distance(x1, y1, x2=0, y2=0)` that returns the distance between the points $(x_1, y_1)$ and
$(x_2, y_2)$. If the second point is left out, it is the origin. You may use `math.sqrt` (the setup cell
already imported `math`).

In [11]:
def distance(x1, y1, x2=0, y2=0):
    """Return the distance between (x1, y1) and (x2, y2)."""
    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

In [12]:
check("1.3 distance(3, 4)", distance(3, 4), 5)
check("1.3 distance(1, 1, 4, 5)", distance(1, 1, 4, 5), 5)
check("1.3 distance(0, 0)", distance(0, 0), 0)
check("1.3 keyword args", distance(x1=-1, y1=-1, x2=2, y2=3), 5)

PASS  1.3 distance(3, 4)
PASS  1.3 distance(1, 1, 4, 5)
PASS  1.3 distance(0, 0)
PASS  1.3 keyword args


### Exercise 1.4: `min_max_mean`
Return a tuple `(smallest, largest, mean)` for a non-empty list of numbers. You may use the built-in
`min`, `max`, `sum`, and `len` here.

In [13]:
def min_max_mean(xs):
    """Return (min, max, mean) of the list xs."""
    return min(xs), max(xs), sum(xs) / len(xs)

In [14]:
check("1.4 min_max_mean([3, 1, 4, 1, 5])", min_max_mean([3, 1, 4, 1, 5]), (1, 5, 2.8))
check("1.4 min_max_mean([7])", min_max_mean([7]), (7, 7, 7.0))
check("1.4 min_max_mean([-2, 2])", min_max_mean([-2, 2]), (-2, 2, 0.0))
check("1.4 returns a tuple", lambda: type(min_max_mean([1, 2])), tuple)
check("1.4 unpacking works", lambda: min_max_mean([10, 20, 60])[2], 30)

PASS  1.4 min_max_mean([3, 1, 4, 1, 5])
PASS  1.4 min_max_mean([7])
PASS  1.4 min_max_mean([-2, 2])
PASS  1.4 returns a tuple
PASS  1.4 unpacking works


### Exercise 1.5: Fix `average_of_three`
The function below is supposed to give back the average of its three arguments, but the test fails.
Run the test first to see what happens, then fix the function.

In [15]:
def average_of_three(a, b, c):
    """Return the average of a, b, and c."""
    total = a + b + c
    return total / 3

In [16]:
check("1.5 average_of_three(1, 2, 3)", average_of_three(1, 2, 3), 2.0)
check("1.5 average_of_three(0, 0, 1)", average_of_three(0, 0, 1), 1 / 3)

PASS  1.5 average_of_three(1, 2, 3)
PASS  1.5 average_of_three(0, 0, 1)


### Exercise 1.6: `letter_grade`
Write `letter_grade(score, cutoffs=(90, 80, 70, 60))`. The four numbers in `cutoffs` are the lowest
scores that earn an A, B, C, and D. Anything below the last cutoff is an F. Return the letter as a
string. A teacher with a different scale can pass in their own cutoffs.

Hint: you can unpack the tuple with `a, b, c, d = cutoffs`.

In [17]:
def letter_grade(score, cutoffs=(90, 80, 70, 60)):
    """Return "A", "B", "C", "D", or "F"."""
    a, b, c, d = cutoffs
    if score >= a:
        return "A"
    if score >= b:
        return "B"
    if score >= c:
        return "C"
    if score >= d:
        return "D"
    return "F"

In [18]:
check("1.6 letter_grade(95)", letter_grade(95), "A")
check("1.6 letter_grade(90)", letter_grade(90), "A")
check("1.6 letter_grade(89.9)", letter_grade(89.9), "B")
check("1.6 letter_grade(72)", letter_grade(72), "C")
check("1.6 letter_grade(60)", letter_grade(60), "D")
check("1.6 letter_grade(59)", letter_grade(59), "F")
check("1.6 custom cutoffs", letter_grade(85, cutoffs=(85, 75, 65, 55)), "A")

PASS  1.6 letter_grade(95)
PASS  1.6 letter_grade(90)
PASS  1.6 letter_grade(89.9)
PASS  1.6 letter_grade(72)
PASS  1.6 letter_grade(60)
PASS  1.6 letter_grade(59)
PASS  1.6 custom cutoffs


## Section 2: Lists and Ranges

A list holds a sequence of values in order. Indexing starts at 0, and negative indices count from the
end, so `xs[-1]` is the last element.

In [19]:
xs = [10, 20, 30, 40, 50]
print(xs[0], xs[2], xs[-1], xs[-2])
print(len(xs))

10 30 50 40
5


**Slicing** pulls out a piece of a list: `xs[start:stop:step]`. The start is included and the stop is
not. Any of the three can be left out. A slice is a new list, so changing it does not change `xs`.

In [20]:
xs = [10, 20, 30, 40, 50]
print(xs[1:4])     # elements 1, 2, 3
print(xs[:2])      # first two
print(xs[2:])      # from index 2 to the end
print(xs[-2:])     # last two
print(xs[::2])     # every other element
print(xs[::-1])    # reversed

[20, 30, 40]
[10, 20]
[30, 40, 50]
[40, 50]
[10, 30, 50]
[50, 40, 30, 20, 10]


**Changing a list.** `append` adds one item to the end, `extend` adds every item from another list,
`insert` puts an item at a position, and `pop` removes and returns an item. `+` makes a new list by
joining two lists, and `*` repeats a list. `in` tests membership.

In [21]:
xs = [1, 2, 3]
xs.append(4)
print(xs)
xs.extend([5, 6])
print(xs)
xs.append([7, 8])        # careful: this appends ONE item, which is itself a list
print(xs, len(xs))
last = xs.pop()
print(last, xs)
print([1, 2] + [3], [0] * 4)
print(3 in xs, 99 in xs)

[1, 2, 3, 4]
[1, 2, 3, 4, 5, 6]
[1, 2, 3, 4, 5, 6, [7, 8]] 7
[7, 8] [1, 2, 3, 4, 5, 6]
[1, 2, 3] [0, 0, 0, 0]
True False


**`range`** produces a sequence of integers, again with the stop excluded. Wrap it in `list(...)` if you
want to see the values.

In [22]:
print(list(range(5)))          # 0..4
print(list(range(2, 8)))       # 2..7
print(list(range(0, 20, 5)))   # step of 5
print(list(range(10, 0, -3)))  # counting down
print(list(range(3, 3)))       # empty

[0, 1, 2, 3, 4]
[2, 3, 4, 5, 6, 7]
[0, 5, 10, 15]
[10, 7, 4, 1]
[]


**`sorted` vs `.sort()`.** `sorted(xs)` returns a new sorted list and leaves `xs` alone. `xs.sort()`
sorts `xs` in place and returns `None`, so `ys = xs.sort()` is a bug.

In [23]:
xs = [3, 1, 2]
ys = sorted(xs)
print(xs, ys)
zs = xs.sort()
print(xs, zs)
print(sorted(xs, reverse=True))

[3, 1, 2] [1, 2, 3]
[1, 2, 3] None
[3, 2, 1]


**Aliasing.** Assigning a list to a new name does *not* copy it. Both names refer to the same list, so a
change through one name shows up through the other. Use `.copy()` (or `xs[:]`) when you want a real
copy.

In [24]:
a = [1, 2, 3]
b = a           # same list, two names
c = a.copy()    # a separate list
a.append(4)
print("a:", a)
print("b:", b)
print("c:", c)

a: [1, 2, 3, 4]
b: [1, 2, 3, 4]
c: [1, 2, 3]


**List comprehensions** build a list from a loop in one line: `[expression for item in sequence]`, with an
optional `if` filter at the end.

In [25]:
squares = [x ** 2 for x in range(6)]
print(squares)
evens = [x for x in range(20) if x % 2 == 0]
print(evens)
words = ["apple", "Banana", "cherry"]
print([w.upper() for w in words])
print([len(w) for w in words if w[0].islower()])

[0, 1, 4, 9, 16, 25]
[0, 2, 4, 6, 8, 10, 12, 14, 16, 18]
['APPLE', 'BANANA', 'CHERRY']
[5, 6]


**Lists of lists** are how we represent a matrix in plain Python: a list of rows. `M[i][j]` is row `i`,
column `j`. There is a trap here: `[[0] * 3] * 2` looks like a 2-by-3 matrix of zeros, but it is the
*same* row repeated twice, so changing one row changes both. Build each row separately instead, for
example with a comprehension.

In [26]:
M = [[1, 2, 3],
     [4, 5, 6]]
print(M[1][2])          # row 1, column 2
print(len(M), len(M[0]))  # number of rows, number of columns

bad = [[0] * 3] * 2
bad[0][0] = 99
print("bad: ", bad)      # both rows changed!

good = [[0] * 3 for _ in range(2)]
good[0][0] = 99
print("good:", good)

6
2 3
bad:  [[99, 0, 0], [99, 0, 0]]
good: [[99, 0, 0], [0, 0, 0]]


### Exercise 2.1: `evens_up_to`
Return a list of the even numbers from 0 up to and including `n`. Use `range`. If `n` is negative,
return an empty list.

In [27]:
def evens_up_to(n):
    """Return [0, 2, 4, ...] up to and including n."""
    return list(range(0, n + 1, 2))

In [28]:
check("2.1 evens_up_to(10)", evens_up_to(10), [0, 2, 4, 6, 8, 10])
check("2.1 evens_up_to(7)", evens_up_to(7), [0, 2, 4, 6])
check("2.1 evens_up_to(0)", evens_up_to(0), [0])
check("2.1 evens_up_to(-1)", evens_up_to(-1), [])

PASS  2.1 evens_up_to(10)
PASS  2.1 evens_up_to(7)
PASS  2.1 evens_up_to(0)
PASS  2.1 evens_up_to(-1)


### Exercise 2.2: `last_k`
Return a new list with the last `k` elements of `xs`. If `k` is larger than the list, return the whole
list. If `k` is 0, return an empty list.

Before you start, try `[1, 2, 3][-0:]` in a scratch cell. It is not what you might guess.

In [29]:
def last_k(xs, k):
    """Return the last k elements of xs."""
    if k <= 0:
        return []
    return xs[-k:]

In [30]:
check("2.2 last_k([1,2,3,4,5], 2)", last_k([1, 2, 3, 4, 5], 2), [4, 5])
check("2.2 last_k([1,2,3,4,5], 0)", last_k([1, 2, 3, 4, 5], 0), [])
check("2.2 last_k([1,2,3,4,5], 10)", last_k([1, 2, 3, 4, 5], 10), [1, 2, 3, 4, 5])
check("2.2 last_k([], 3)", last_k([], 3), [])

PASS  2.2 last_k([1,2,3,4,5], 2)
PASS  2.2 last_k([1,2,3,4,5], 0)
PASS  2.2 last_k([1,2,3,4,5], 10)
PASS  2.2 last_k([], 3)


### Exercise 2.3: `squares_of_odds`
Use a list comprehension to return the squares of the odd numbers in `xs`, in their original order.

In [31]:
def squares_of_odds(xs):
    """Return [x**2 for each odd x in xs]."""
    return [x ** 2 for x in xs if x % 2 == 1]

In [32]:
check("2.3 squares_of_odds([1,2,3,4,5])", squares_of_odds([1, 2, 3, 4, 5]), [1, 9, 25])
check("2.3 squares_of_odds([])", squares_of_odds([]), [])
check("2.3 squares_of_odds([-3, 2])", squares_of_odds([-3, 2]), [9])
check("2.3 squares_of_odds([2, 4])", squares_of_odds([2, 4]), [])

PASS  2.3 squares_of_odds([1,2,3,4,5])
PASS  2.3 squares_of_odds([])
PASS  2.3 squares_of_odds([-3, 2])
PASS  2.3 squares_of_odds([2, 4])


### Exercise 2.4: `middle`
Return the middle element of a non-empty list. If the list has an even length, there are two middle
elements; return their average. (This is just about positions, so do not sort.)

Hint: integer division `//` is your friend for computing an index.

In [33]:
def middle(xs):
    """Return the middle element of xs, or the average of the two middle elements."""
    n = len(xs)
    if n % 2 == 1:
        return xs[n // 2]
    return (xs[n // 2 - 1] + xs[n // 2]) / 2

In [34]:
check("2.4 middle([1, 2, 3])", middle([1, 2, 3]), 2)
check("2.4 middle([1, 2, 3, 4])", middle([1, 2, 3, 4]), 2.5)
check("2.4 middle([7])", middle([7]), 7)
check("2.4 middle([9, 1, 5, 3, 8])", middle([9, 1, 5, 3, 8]), 5)

PASS  2.4 middle([1, 2, 3])
PASS  2.4 middle([1, 2, 3, 4])
PASS  2.4 middle([7])
PASS  2.4 middle([9, 1, 5, 3, 8])


### Exercise 2.5: `sorted_desc_copy`
Return a new list with the elements of `xs` sorted from largest to smallest. The original list must
not change.

In [35]:
def sorted_desc_copy(xs):
    """Return a new list sorted largest to smallest. Do not change xs."""
    return sorted(xs, reverse=True)

In [36]:
check("2.5 sorted_desc_copy([3, 1, 2])", sorted_desc_copy([3, 1, 2]), [3, 2, 1])
check("2.5 sorted_desc_copy([])", sorted_desc_copy([]), [])

def _t():
    xs = [5, 9, 1]
    sorted_desc_copy(xs)
    return xs
check("2.5 does not change xs", _t, [5, 9, 1])

PASS  2.5 sorted_desc_copy([3, 1, 2])
PASS  2.5 sorted_desc_copy([])
PASS  2.5 does not change xs


### Exercise 2.6: `zeros_matrix`
Return an `m` by `n` matrix of zeros as a list of `m` rows, each a list of `n` zeros. The rows must be
separate lists (see the trap above).

In [37]:
def zeros_matrix(m, n):
    """Return an m-by-n list of lists of zeros."""
    return [[0] * n for _ in range(m)]

In [38]:
check("2.6 zeros_matrix(2, 3)", zeros_matrix(2, 3), [[0, 0, 0], [0, 0, 0]])
check("2.6 zeros_matrix(1, 1)", zeros_matrix(1, 1), [[0]])
check("2.6 zeros_matrix(0, 3)", zeros_matrix(0, 3), [])

def _t():
    M = zeros_matrix(2, 3)
    M[0][1] = 5
    return M
check("2.6 rows are separate lists", _t, [[0, 5, 0], [0, 0, 0]])

PASS  2.6 zeros_matrix(2, 3)
PASS  2.6 zeros_matrix(1, 1)
PASS  2.6 zeros_matrix(0, 3)
PASS  2.6 rows are separate lists


### Exercise 2.7: `transpose`
Return the transpose of a matrix stored as a list of lists: row `i` of the result is column `i` of `M`.
Do not change `M`. You can use loops or a nested comprehension. You can assume `M` has at least one
row.

In [39]:
def transpose(M):
    """Return the transpose of the list-of-lists matrix M."""
    rows, cols = len(M), len(M[0])
    return [[M[i][j] for i in range(rows)] for j in range(cols)]

In [40]:
check("2.7 transpose 2x3", transpose([[1, 2, 3], [4, 5, 6]]), [[1, 4], [2, 5], [3, 6]])
check("2.7 transpose 1x1", transpose([[1]]), [[1]])
check("2.7 transpose 1x2", transpose([[1, 2]]), [[1], [2]])

def _t():
    M = [[1, 2], [3, 4]]
    transpose(M)
    return M
check("2.7 does not change M", _t, [[1, 2], [3, 4]])

PASS  2.7 transpose 2x3
PASS  2.7 transpose 1x1
PASS  2.7 transpose 1x2
PASS  2.7 does not change M


### Exercise 2.8: `flatten`
Turn a list of lists into one flat list, keeping the order. Some of the inner lists may be empty.

In [41]:
def flatten(M):
    """Return one list containing every element of every row of M, in order."""
    return [x for row in M for x in row]

In [42]:
check("2.8 flatten", flatten([[1, 2], [3], [], [4, 5, 6]]), [1, 2, 3, 4, 5, 6])
check("2.8 flatten([])", flatten([]), [])
check("2.8 flatten([[7]])", flatten([[7]]), [7])

PASS  2.8 flatten
PASS  2.8 flatten([])
PASS  2.8 flatten([[7]])


## Section 3: Loops

A `for` loop runs its body once for each item in a sequence. Loop over the items directly when you
only need the values. Use `range(len(xs))` when you need the positions.

In [43]:
names = ["Ada", "Grace", "Alan"]
for name in names:
    print("hello", name)

for i in range(len(names)):
    print(i, names[i])

hello Ada
hello Grace
hello Alan
0 Ada
1 Grace
2 Alan


`enumerate` gives you the index and the item together, and `zip` walks through two (or more) lists in
step. Both are usually cleaner than indexing by hand.

In [44]:
names = ["Ada", "Grace", "Alan"]
scores = [92, 88, 95]
for i, name in enumerate(names):
    print(i, name)
for name, score in zip(names, scores):
    print(f"{name} scored {score}")

0 Ada
1 Grace
2 Alan
Ada scored 92
Grace scored 88
Alan scored 95


**The accumulator pattern** is the basic shape of most loop code: start a variable before the loop,
update it inside, use it after.

In [45]:
xs = [4, 8, 15, 16, 23, 42]
total = 0
for x in xs:
    total += x
print("sum:", total)

biggest = xs[0]
for x in xs[1:]:
    if x > biggest:
        biggest = x
print("max:", biggest)

sum: 108
max: 42


A `while` loop keeps going as long as its condition is true. Use it when you do not know ahead of time
how many steps you need. `break` leaves a loop immediately and `continue` skips to the next pass.

In [46]:
n = 100
steps = 0
while n > 1:
    n = n // 2
    steps += 1
print("halvings:", steps)

for x in [3, -1, 4, -1, 5, 9, 2]:
    if x < 0:
        continue      # skip negatives
    if x > 8:
        break         # stop entirely
    print(x)

halvings: 6
3
4
5


**Nested loops** run the inner loop all the way through for each pass of the outer loop. This is how we
will build matrix code below.

In [47]:
for i in range(1, 4):
    row = []
    for j in range(1, 5):
        row.append(i * j)
    print(row)

[1, 2, 3, 4]
[2, 4, 6, 8]
[3, 6, 9, 12]


### Exercise 3.1: `count_vowels`
Count the vowels (a, e, i, o, u) in a string, ignoring case. A `for` loop over a string visits each
character.

In [48]:
def count_vowels(s):
    """Return the number of vowels in s (case-insensitive)."""
    count = 0
    for ch in s.lower():
        if ch in "aeiou":
            count += 1
    return count

In [49]:
check("3.1 count_vowels('hello')", count_vowels("hello"), 2)
check("3.1 count_vowels('AEIOU xyz')", count_vowels("AEIOU xyz"), 5)
check("3.1 count_vowels('')", count_vowels(""), 0)
check("3.1 count_vowels('rhythm')", count_vowels("rhythm"), 0)

PASS  3.1 count_vowels('hello')
PASS  3.1 count_vowels('AEIOU xyz')
PASS  3.1 count_vowels('')
PASS  3.1 count_vowels('rhythm')


### Exercise 3.2: `argmax`
Return the *index* of the largest element of a non-empty list. If the largest value appears more than
once, return the first index. Do not use `max` or `.index`.

In [50]:
def argmax(xs):
    """Return the index of the first largest element of xs."""
    best = 0
    for i in range(1, len(xs)):
        if xs[i] > xs[best]:
            best = i
    return best

In [51]:
check("3.2 argmax([3, 7, 2, 7])", argmax([3, 7, 2, 7]), 1)
check("3.2 argmax([5])", argmax([5]), 0)
check("3.2 argmax([-5, -2, -9])", argmax([-5, -2, -9]), 1)
check("3.2 argmax([1, 2, 3])", argmax([1, 2, 3]), 2)

PASS  3.2 argmax([3, 7, 2, 7])
PASS  3.2 argmax([5])
PASS  3.2 argmax([-5, -2, -9])
PASS  3.2 argmax([1, 2, 3])


### Exercise 3.3: `running_sum`
Return a list where element `i` is the sum of `xs[0]` through `xs[i]`.

In [52]:
def running_sum(xs):
    """Return the list of partial sums of xs."""
    result = []
    total = 0
    for x in xs:
        total += x
        result.append(total)
    return result

In [53]:
check("3.3 running_sum([1, 2, 3, 4])", running_sum([1, 2, 3, 4]), [1, 3, 6, 10])
check("3.3 running_sum([])", running_sum([]), [])
check("3.3 running_sum([5, -5, 5])", running_sum([5, -5, 5]), [5, 0, 5])

PASS  3.3 running_sum([1, 2, 3, 4])
PASS  3.3 running_sum([])
PASS  3.3 running_sum([5, -5, 5])


### Exercise 3.4: `first_index_over`
Return the index of the first element of `xs` that is strictly greater than `threshold`, or `-1` if
there is none. Stop looking as soon as you find one.

In [54]:
def first_index_over(xs, threshold):
    """Return the index of the first x > threshold, or -1."""
    for i, x in enumerate(xs):
        if x > threshold:
            return i
    return -1

In [55]:
check("3.4 first_index_over([1, 5, 2, 8], 4)", first_index_over([1, 5, 2, 8], 4), 1)
check("3.4 first_index_over([1, 2], 10)", first_index_over([1, 2], 10), -1)
check("3.4 first_index_over([], 0)", first_index_over([], 0), -1)
check("3.4 strictly greater", first_index_over([4, 4, 5], 4), 2)

PASS  3.4 first_index_over([1, 5, 2, 8], 4)
PASS  3.4 first_index_over([1, 2], 10)
PASS  3.4 first_index_over([], 0)
PASS  3.4 strictly greater


### Exercise 3.5: `collatz_steps`
Start with a positive integer `n`. If it is even, halve it; if it is odd, replace it with `3n + 1`.
Return how many steps it takes to reach 1. (For `n = 6` the sequence is 6, 3, 10, 5, 16, 8, 4, 2, 1,
which is 8 steps.) This is a job for a `while` loop.

In [56]:
def collatz_steps(n):
    """Return the number of Collatz steps from n down to 1."""
    steps = 0
    while n != 1:
        if n % 2 == 0:
            n = n // 2
        else:
            n = 3 * n + 1
        steps += 1
    return steps

In [57]:
check("3.5 collatz_steps(1)", collatz_steps(1), 0)
check("3.5 collatz_steps(6)", collatz_steps(6), 8)
check("3.5 collatz_steps(27)", collatz_steps(27), 111)

PASS  3.5 collatz_steps(1)
PASS  3.5 collatz_steps(6)
PASS  3.5 collatz_steps(27)


### Exercise 3.6: `mse`
The mean squared error between true values $y$ and predictions $\hat y$ is

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n} (y_i - \hat y_i)^2.$$

Write it with a loop (`zip` is handy). If the two lists have different lengths, raise an error with
`raise ValueError("lengths differ")`.

In [58]:
def mse(y_true, y_pred):
    """Return the mean squared error between two equal-length lists."""
    if len(y_true) != len(y_pred):
        raise ValueError("lengths differ")
    total = 0
    for y, yhat in zip(y_true, y_pred):
        total += (y - yhat) ** 2
    return total / len(y_true)

In [59]:
check("3.6 mse perfect", mse([1, 2, 3], [1, 2, 3]), 0.0)
check("3.6 mse([1,2,3], [2,2,5])", mse([1, 2, 3], [2, 2, 5]), 5 / 3)
check("3.6 mse([0], [4])", mse([0], [4]), 16)
check_raises("3.6 mse raises on length mismatch", lambda: mse([1, 2], [1]), ValueError)

PASS  3.6 mse perfect
PASS  3.6 mse([1,2,3], [2,2,5])
PASS  3.6 mse([0], [4])
PASS  3.6 mse raises on length mismatch


### Exercise 3.7: `dot`
The dot product of two vectors of the same length is $u \cdot v = \sum_i u_i v_i$. Write it with a
loop. Raise a `ValueError` if the lengths differ. The dot product of two empty lists is 0.

In [60]:
def dot(u, v):
    """Return the dot product of lists u and v."""
    if len(u) != len(v):
        raise ValueError("lengths differ")
    total = 0
    for i in range(len(u)):
        total += u[i] * v[i]
    return total

In [61]:
check("3.7 dot([1,2,3], [4,5,6])", dot([1, 2, 3], [4, 5, 6]), 32)
check("3.7 dot([], [])", dot([], []), 0)
check("3.7 dot([1, -1], [1, 1])", dot([1, -1], [1, 1]), 0)
check_raises("3.7 dot raises on length mismatch", lambda: dot([1, 2], [1, 2, 3]), ValueError)

PASS  3.7 dot([1,2,3], [4,5,6])
PASS  3.7 dot([], [])
PASS  3.7 dot([1, -1], [1, 1])
PASS  3.7 dot raises on length mismatch


### Exercise 3.8: `mat_vec`
Multiply a matrix `M` (a list of rows) by a vector `v` (a list). Entry `i` of the result is the dot
product of row `i` of `M` with `v`. You can call your `dot` function.

In [62]:
def mat_vec(M, v):
    """Return the matrix-vector product M v as a list."""
    return [dot(row, v) for row in M]

In [63]:
check("3.8 mat_vec 2x2", mat_vec([[1, 2], [3, 4]], [1, 1]), [3, 7])
check("3.8 mat_vec 2x3", mat_vec([[1, 0, 0], [0, 2, 0]], [3, 4, 5]), [3, 8])
check("3.8 mat_vec 1x1", mat_vec([[5]], [2]), [10])

PASS  3.8 mat_vec 2x2
PASS  3.8 mat_vec 2x3
PASS  3.8 mat_vec 1x1


### Exercise 3.9: `identity`
Return the `n` by `n` identity matrix as a list of lists: 1 on the diagonal, 0 everywhere else.

In [64]:
def identity(n):
    """Return the n-by-n identity matrix."""
    I = [[0] * n for _ in range(n)]
    for i in range(n):
        I[i][i] = 1
    return I

In [65]:
check("3.9 identity(1)", identity(1), [[1]])
check("3.9 identity(3)", identity(3), [[1, 0, 0], [0, 1, 0], [0, 0, 1]])

def _t():
    I = identity(2)
    I[0][1] = 7
    return I
check("3.9 rows are separate lists", _t, [[1, 7], [0, 1]])

PASS  3.9 identity(1)
PASS  3.9 identity(3)
PASS  3.9 rows are separate lists


### Exercise 3.10: `mat_mul`
Multiply matrix `A` (size $n \times m$) by matrix `B` (size $m \times p$). The result $C$ is
$n \times p$ with

$$C_{ij} = \sum_{k=0}^{m-1} A_{ik} B_{kj}.$$

Use three nested loops (start from `zeros_matrix`). If the number of columns of `A` does not match the
number of rows of `B`, raise a `ValueError`.

In [66]:
def mat_mul(A, B):
    """Return the matrix product A B, with matrices stored as lists of rows."""
    n, m, p = len(A), len(B), len(B[0])
    if len(A[0]) != m:
        raise ValueError("inner dimensions do not match")
    C = zeros_matrix(n, p)
    for i in range(n):
        for j in range(p):
            for k in range(m):
                C[i][j] += A[i][k] * B[k][j]
    return C

In [67]:
check("3.10 mat_mul 2x2", mat_mul([[1, 2], [3, 4]], [[5, 6], [7, 8]]), [[19, 22], [43, 50]])
check("3.10 mat_mul (1x3)(3x1)", mat_mul([[1, 2, 3]], [[1], [2], [3]]), [[14]])
check("3.10 mat_mul (2x1)(1x2)", mat_mul([[1], [2]], [[3, 4]]), [[3, 4], [6, 8]])
check("3.10 A times identity", lambda: mat_mul([[2, 3], [4, 5]], identity(2)), [[2, 3], [4, 5]])
check("3.10 order matters", mat_mul([[5, 6], [7, 8]], [[1, 2], [3, 4]]), [[23, 34], [31, 46]])
check_raises("3.10 mat_mul raises on bad shapes", lambda: mat_mul([[1, 2]], [[1, 2]]), ValueError)

PASS  3.10 mat_mul 2x2
PASS  3.10 mat_mul (1x3)(3x1)
PASS  3.10 mat_mul (2x1)(1x2)
PASS  3.10 A times identity
PASS  3.10 order matters
PASS  3.10 mat_mul raises on bad shapes


## Section 4: Dictionaries and Strings

A dictionary maps *keys* to *values*. Looking up a key that is not there raises a `KeyError`; the
`.get` method returns a default instead.

In [68]:
ages = {"Ada": 36, "Alan": 41}
ages["Grace"] = 85          # add a new key
ages["Ada"] = 37            # change an existing one
print(ages)
print(ages["Alan"])
print("Grace" in ages, "Linus" in ages)
print(ages.get("Linus"), ages.get("Linus", 0))
print(len(ages), list(ages.keys()), list(ages.values()))
for name, age in ages.items():
    print(name, age)

{'Ada': 37, 'Alan': 41, 'Grace': 85}
41
True False
None 0
3 ['Ada', 'Alan', 'Grace'] [37, 41, 85]
Ada 37
Alan 41
Grace 85


**Counting with a dictionary** is a pattern you will use constantly. `d.get(key, 0) + 1` handles the
first time you see a key.

In [69]:
letters = "mississippi"
counts = {}
for ch in letters:
    counts[ch] = counts.get(ch, 0) + 1
print(counts)

{'m': 1, 'i': 4, 's': 4, 'p': 2}


**Strings** have many useful methods. They never change the original string (strings are immutable);
they return new ones. f-strings format values into text, and a format spec after a colon controls
things like decimal places.

In [70]:
s = "  The Quick Brown Fox.  "
print(repr(s.strip()))
print(s.lower())
print(s.split())                    # split on whitespace
print("a,b,,c".split(","))
print("-".join(["x", "y", "z"]))
print("fox...".strip("."))

pi = 3.14159265
name = "pi"
print(f"{name} is about {pi:.2f}")
print(f"{name:>6}|{pi:8.3f}|")    # right-align in 6 characters, 8 wide with 3 decimals

'The Quick Brown Fox.'
  the quick brown fox.  
['The', 'Quick', 'Brown', 'Fox.']
['a', 'b', '', 'c']
x-y-z
fox
pi is about 3.14
    pi|   3.142|


### Exercise 4.1: `count_values`
Return a dictionary mapping each distinct value in `xs` to how many times it appears.

In [71]:
def count_values(xs):
    """Return a dict of value -> count."""
    counts = {}
    for x in xs:
        counts[x] = counts.get(x, 0) + 1
    return counts

In [72]:
check("4.1 count_values strings", count_values(["a", "b", "a"]), {"a": 2, "b": 1})
check("4.1 count_values([])", count_values([]), {})
check("4.1 count_values numbers", count_values([1, 1, 1, 2]), {1: 3, 2: 1})

PASS  4.1 count_values strings
PASS  4.1 count_values([])
PASS  4.1 count_values numbers


### Exercise 4.2: `word_freq`
Return a dictionary of word counts for a piece of text. Make everything lowercase, split on whitespace,
and strip the punctuation characters `.,!?;:` from both ends of each word. Skip any word that becomes
empty after stripping.

In [73]:
def word_freq(text):
    """Return a dict of lowercase word -> count."""
    counts = {}
    for word in text.lower().split():
        word = word.strip(".,!?;:")
        if word:
            counts[word] = counts.get(word, 0) + 1
    return counts

In [74]:
check("4.2 word_freq sentence", word_freq("The cat and the hat."),
      {"the": 2, "cat": 1, "and": 1, "hat": 1})
check("4.2 word_freq case and punctuation", word_freq("Hi! hi, HI."), {"hi": 3})
check("4.2 word_freq('')", word_freq(""), {})
check("4.2 word_freq lone punctuation", word_freq("wait ... what?!"), {"wait": 1, "what": 1})

PASS  4.2 word_freq sentence
PASS  4.2 word_freq case and punctuation
PASS  4.2 word_freq('')
PASS  4.2 word_freq lone punctuation


### Exercise 4.3: `most_common`
Given a dictionary of counts, return the key with the largest count. If there is a tie, return the one
that comes first in the dictionary. Return `None` for an empty dictionary.

In [75]:
def most_common(counts):
    """Return the key with the largest value, or None if counts is empty."""
    best_key = None
    for key, value in counts.items():
        if best_key is None or value > counts[best_key]:
            best_key = key
    return best_key

In [76]:
check("4.3 most_common", most_common({"a": 2, "b": 5, "c": 1}), "b")
check("4.3 most_common tie", most_common({"a": 2, "b": 5, "c": 5}), "b")
check("4.3 most_common({})", most_common({}), None)
check("4.3 with word_freq", lambda: most_common(word_freq("to be or not to be to")), "to")

PASS  4.3 most_common
PASS  4.3 most_common tie
PASS  4.3 most_common({})
PASS  4.3 with word_freq


### Exercise 4.4: `label_encode`
Machine learning models need numbers, not strings. Given a list of category names, assign each
distinct category an integer in the order it first appears (first new category is 0, next is 1, and
so on). Return a tuple `(codes, mapping)` where `codes` is the list of integers and `mapping` is the
dictionary from category to integer.

In [77]:
def label_encode(categories):
    """Return (codes, mapping) for a list of category names."""
    mapping = {}
    codes = []
    for c in categories:
        if c not in mapping:
            mapping[c] = len(mapping)
        codes.append(mapping[c])
    return codes, mapping

In [78]:
check("4.4 label_encode", label_encode(["red", "green", "red", "blue"]),
      ([0, 1, 0, 2], {"red": 0, "green": 1, "blue": 2}))
check("4.4 label_encode([])", label_encode([]), ([], {}))
check("4.4 label_encode one category", label_encode(["x", "x"]), ([0, 0], {"x": 0}))

PASS  4.4 label_encode
PASS  4.4 label_encode([])
PASS  4.4 label_encode one category


### Exercise 4.5: `invert_dict`
Return a new dictionary with keys and values swapped. You can assume all the values are different.

In [79]:
def invert_dict(d):
    """Return {value: key for each key, value in d}."""
    return {value: key for key, value in d.items()}

In [80]:
check("4.5 invert_dict", invert_dict({"a": 1, "b": 2}), {1: "a", 2: "b"})
check("4.5 invert_dict({})", invert_dict({}), {})
check("4.5 undo label_encode", lambda: invert_dict(label_encode(["cat", "dog"])[1]), {0: "cat", 1: "dog"})

PASS  4.5 invert_dict
PASS  4.5 invert_dict({})
PASS  4.5 undo label_encode


### Exercise 4.6: `format_score`
Return a string like `"Ada: 92.50"`: the name, a colon and a space, and the score with exactly two
decimal places. Use an f-string.

In [81]:
def format_score(name, score):
    """Return "name: score" with the score to two decimal places."""
    return f"{name}: {score:.2f}"

In [82]:
check("4.6 format_score('Ada', 92.5)", format_score("Ada", 92.5), "Ada: 92.50")
check("4.6 format_score('Bob', 7)", format_score("Bob", 7), "Bob: 7.00")
check("4.6 format_score('Cy', 1/3)", format_score("Cy", 1 / 3), "Cy: 0.33")

PASS  4.6 format_score('Ada', 92.5)
PASS  4.6 format_score('Bob', 7)
PASS  4.6 format_score('Cy', 1/3)


## Section 5: Reading Errors and Fixing Bugs

When Python hits an error it prints a *traceback*. Read it from the bottom up: the last line tells you
the kind of error and a message, and the lines just above it point at the line of your code where it
happened. The cell below shows a traceback without stopping the notebook.

In [83]:
import traceback

def get_fourth(xs):
    return xs[3]

try:
    get_fourth([10, 20, 30])
except Exception:
    traceback.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_322/3725389591.py", line 7, in <module>
    get_fourth([10, 20, 30])
    ~~~~~~~~~~^^^^^^^^^^^^^^
  File "/tmp/ipykernel_322/3725389591.py", line 4, in get_fourth
    return xs[3]
           ~~^^^
IndexError: list index out of range


The errors you will see most often:

| Error | Usual cause |
|---|---|
| `IndexError` | an index past the end of a list, often an off-by-one in a `range` |
| `KeyError` | looking up a dictionary key that is not there (use `in` or `.get`) |
| `TypeError` | mixing types, like `[1, 2] + 3`, or calling something with the wrong number of arguments |
| `NameError` | a misspelled variable, or a cell you have not run yet |
| `ZeroDivisionError` | dividing by a length or count that can be zero |
| `AttributeError` | calling a method on the wrong type, often on a `None` that came from a function with no `return` |

The worst bugs do not raise any error at all: an off-by-one in a loop, a `return` indented one level
too deep, or changing a list while you loop over it. Those just give wrong answers, which is why we
test.

Each function below has at least one bug. Run its test, read what goes wrong, then fix the function.
Do not rewrite the function from scratch; find the actual mistake.

### Exercise 5.1: `sum_to`

In [84]:
def sum_to(n):
    """Return 1 + 2 + ... + n (0 if n is 0)."""
    total = 0
    for i in range(1, n + 1):
        total += i
    return total

In [85]:
check("5.1 sum_to(4)", sum_to(4), 10)
check("5.1 sum_to(1)", sum_to(1), 1)
check("5.1 sum_to(0)", sum_to(0), 0)
check("5.1 sum_to(100)", sum_to(100), 5050)

PASS  5.1 sum_to(4)
PASS  5.1 sum_to(1)
PASS  5.1 sum_to(0)
PASS  5.1 sum_to(100)


### Exercise 5.2: `remove_negatives`

In [86]:
def remove_negatives(xs):
    """Return a new list with the negative numbers removed. Do not change xs."""
    result = []
    for x in xs:
        if x >= 0:
            result.append(x)
    return result

In [87]:
check("5.2 remove_negatives([1, -2, 3])", lambda: remove_negatives([1, -2, 3]), [1, 3])
check("5.2 adjacent negatives", lambda: remove_negatives([1, -2, -3, 4]), [1, 4])
check("5.2 all negative", lambda: remove_negatives([-1, -1, -1]), [])

def _t():
    xs = [5, -1, 6]
    remove_negatives(xs)
    return xs
check("5.2 does not change xs", _t, [5, -1, 6])

PASS  5.2 remove_negatives([1, -2, 3])
PASS  5.2 adjacent negatives
PASS  5.2 all negative
PASS  5.2 does not change xs


### Exercise 5.3: `average_score`

In [88]:
def average_score(scores, name):
    """scores maps names to lists of scores.
    Return the average score for name, or None if name is missing or has no scores."""
    if name not in scores or len(scores[name]) == 0:
        return None
    return sum(scores[name]) / len(scores[name])

In [89]:
gradebook = {"ada": [90, 80], "bob": [70, 80, 90, 100], "cy": []}
check("5.3 average_score ada", lambda: average_score(gradebook, "ada"), 85)
check("5.3 average_score bob", lambda: average_score(gradebook, "bob"), 85)
check("5.3 no scores", lambda: average_score(gradebook, "cy"), None)
check("5.3 missing name", lambda: average_score(gradebook, "dee"), None)

PASS  5.3 average_score ada
PASS  5.3 average_score bob
PASS  5.3 no scores
PASS  5.3 missing name


### Exercise 5.4: `count_above_mean`

In [90]:
def count_above_mean(xs):
    """Return how many elements are strictly greater than the mean of xs (0 for an empty list)."""
    if len(xs) == 0:
        return 0
    mean = sum(xs) / len(xs)
    count = 0
    for x in xs:
        if x > mean:
            count += 1
    return count

In [91]:
check("5.4 count_above_mean([1, 2, 3, 4])", lambda: count_above_mean([1, 2, 3, 4]), 2)
check("5.4 count_above_mean([5, 5, 5])", lambda: count_above_mean([5, 5, 5]), 0)
check("5.4 count_above_mean([1, 10])", lambda: count_above_mean([1, 10]), 1)
check("5.4 count_above_mean([])", lambda: count_above_mean([]), 0)

PASS  5.4 count_above_mean([1, 2, 3, 4])
PASS  5.4 count_above_mean([5, 5, 5])
PASS  5.4 count_above_mean([1, 10])
PASS  5.4 count_above_mean([])


## Summary

Run the cell below to see how many tests pass. Before submitting, do **Runtime > Restart session and
run all** so the count reflects your final code.

In [92]:
report()

132 / 132 tests passed
